> Legacy/optional workflow. For author SLM-SQL + CSC baseline and CSC LoRA research, use **09 → 10 → 11** and `docs/CSC_LORA.md`. This notebook is not the canonical comparison against the author system.

# 02 — Baseline Evaluation

Run Qwen2.5-Coder 0.5B and 1.5B (base, no fine-tuning) on BIRD dev.
Modes: greedy (E0) and SC@16 (E1). Compare with published numbers.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path


def bootstrap_repo():
    configured = os.environ.get('SLM_PRJ_PATH')
    candidates = [Path(configured).expanduser()] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents,
                       Path('/kaggle/working/SLM-PRJ'),
                       Path('/kaggle/working/SLMSQL_project')])
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'benchmarks').is_dir():
            return candidate
    target = Path('/kaggle/working/SLM-PRJ')
    url = os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git')
    subprocess.run(['git', 'clone', url, str(target)], check=True)
    return target

REPO_DIR = bootstrap_repo()
sys.path.insert(0, str(REPO_DIR))
from scripts.kaggle_utils import prepare_repo_import, resolve_data_dir
REPO_DIR = prepare_repo_import(REPO_DIR)
DATA_DIR = resolve_data_dir(REPO_DIR)

BIRD_DIR = DATA_DIR / 'bird'
RESULTS_DIR = REPO_DIR / 'results' / 'baselines'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
limit = int(os.environ.get('BIRD_LIMIT', '0'))
print(f'Repo: {REPO_DIR}')
print(f'BIRD: {BIRD_DIR}; evaluation: held-out legacy dev; limit={limit}')

In [ ]:
from benchmarks.bird_loader import load_bird_dev, get_db_path, get_schema

all_bird_dev = load_bird_dev(BIRD_DIR)
from research.legacy_protocol import split_legacy_dev
partitions = split_legacy_dev(all_bird_dev, REPO_DIR / 'results/legacy_protocol/data_split.json')
bird_dev = partitions['evaluation']
bird_dev = bird_dev if limit == 0 else bird_dev[:limit]
print(f'BIRD dev loaded: {len(bird_dev)} / {len(all_bird_dev)} questions')


## 1. Load model with vLLM

In [ ]:
from vllm import LLM, SamplingParams

MODEL_NAME = os.environ.get('BASE_MODEL', 'Qwen/Qwen2.5-Coder-1.5B-Instruct')
MAX_MODEL_LEN = int(os.environ.get('MAX_MODEL_LEN', '8192'))
MAX_OUTPUT_TOKENS = int(os.environ.get('MAX_OUTPUT_TOKENS', '512'))
llm = LLM(
    model=MODEL_NAME,
    dtype='float16',
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=float(os.environ.get('GPU_MEMORY_UTILIZATION', '0.85')),
)
tokenizer = llm.get_tokenizer()
print(f'Loaded {MODEL_NAME}')


def generate_batched(prompt_list, params, batch_size=None):
    batch_size = batch_size or int(os.environ.get('GEN_BATCH_SIZE', '8'))
    outputs = []
    for start_idx in range(0, len(prompt_list), batch_size):
        outputs.extend(llm.generate(prompt_list[start_idx:start_idx + batch_size], params))
    return outputs

# Separate model folders prevent the 0.5B/1.5B baseline runs overwriting each other.
MODEL_KEY = MODEL_NAME.rsplit('/', 1)[-1]
RESULTS_DIR = REPO_DIR / 'results' / 'baselines' / MODEL_KEY
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

def make_chat_prompt(question, schema, evidence=''):
    from training.prompt_template import format_sft_messages
    messages = format_sft_messages(question, schema, '', evidence=evidence)[:-1]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

prompts = []
for r in bird_dev:
    schema = get_schema(BIRD_DIR, r['db_id'])
    prompts.append(make_chat_prompt(r['question'], schema, r.get('evidence', '')))

print(f'Built {len(prompts)} prompts')
print(f'Example prompt length: {len(prompts[0])} chars')
prompt_lengths = [len(tokenizer.encode(prompt, add_special_tokens=False)) for prompt in prompts]
longest_idx = max(range(len(prompt_lengths)), key=prompt_lengths.__getitem__)
print(f'Prompt tokens: max={prompt_lengths[longest_idx]}, median={sorted(prompt_lengths)[len(prompt_lengths) // 2]}')
available_input_tokens = MAX_MODEL_LEN - MAX_OUTPUT_TOKENS
too_long = [(r['db_id'], count) for r, count in zip(bird_dev, prompt_lengths) if count > available_input_tokens]
if too_long:
    db_id, token_count = max(too_long, key=lambda item: item[1])
    raise ValueError(
        f'{len(too_long)} prompt(s) exceed the context budget: longest is {token_count} tokens '
        f'for database {db_id}; MAX_MODEL_LEN={MAX_MODEL_LEN} and MAX_OUTPUT_TOKENS={MAX_OUTPUT_TOKENS}. '
        'Increase MAX_MODEL_LEN (up to the model limit) before loading the model, or reduce MAX_OUTPUT_TOKENS.'
    )

## 2. E0 — Greedy decoding

In [ ]:
from core.sql_extractor import extract_sql
from benchmarks.eval_ex import execution_match

greedy_params = SamplingParams(temperature=0.0, max_tokens=MAX_OUTPUT_TOKENS, seed=42)

start = time.time()
outputs = generate_batched(prompts, greedy_params)
elapsed = time.time() - start
print(f'Greedy generation: {elapsed:.1f}s ({elapsed/len(prompts):.2f}s/query)')

greedy_results = []
correct = 0
for i, (r, out) in enumerate(zip(bird_dev, outputs)):
    raw = out.outputs[0].text
    try:
        pred_sql = extract_sql(raw)
    except ValueError:
        pred_sql = ''
    db = get_db_path(BIRD_DIR, r['db_id'])
    from research.exec_runner import _exec_one
    gold = _exec_one(r['gold_sql'], str(db))
    if gold['fingerprint'] is None:
        raise ValueError(f"Gold query failed for {r['db_id']}")
    matched = _exec_one(pred_sql, str(db))['fingerprint'] == gold['fingerprint']
    correct += int(matched)
    greedy_results.append({
        'question': r['question'], 'db_id': r['db_id'],
        'difficulty': r['difficulty'],
        'pred_sql': pred_sql, 'gold_sql': r['gold_sql'],
        'correct': matched,
    })

ex = correct / len(bird_dev)
print(f'E0 Greedy EX: {ex:.4f} ({correct}/{len(bird_dev)})')
print('Paper Table 2 base reference: 28.40%; not a matched greedy reproduction.')

with open(RESULTS_DIR / 'E0_greedy.json', 'w') as f:
    json.dump({'accuracy': ex, 'total': len(bird_dev), 'correct': correct,
               'model': MODEL_NAME, 'elapsed_s': elapsed, 'records': greedy_results,
               'split': 'heldout_legacy_dev', 'seed': 42, 'evaluator': 'bird-set-v1'}, f, indent=2)

## 3. E1 — Self-Consistency @ 16

In [ ]:
from collections import Counter
from core.result_normalizer import fingerprint, requires_order
from core.schema_inspector import open_readonly
import sqlite3

N_SAMPLES = 16
sc_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=MAX_OUTPUT_TOKENS, n=N_SAMPLES, seed=42)

start = time.time()
sc_outputs = generate_batched(prompts, sc_params)
sc_elapsed = time.time() - start
print(f'SC@{N_SAMPLES} generation: {sc_elapsed:.1f}s')

def exec_fingerprint(sql, db_path, timeout=30):
    from research.exec_runner import _exec_one
    return _exec_one(sql, str(db_path), timeout)['fingerprint']

sc_correct = 0
sc_results = []
for i, (r, out) in enumerate(zip(bird_dev, sc_outputs)):
    db = get_db_path(BIRD_DIR, r['db_id'])
    candidates = []
    for o in out.outputs:
        try:
            sql = extract_sql(o.text)
        except ValueError:
            continue
        fp = exec_fingerprint(sql, db)
        if fp is not None:
            candidates.append((sql, fp))

    if candidates:
        vote = Counter(fp for _, fp in candidates)
        best_fp = vote.most_common(1)[0][0]
        best_sql = next(sql for sql, fp in candidates if fp == best_fp)
        gold_fp = exec_fingerprint(r['gold_sql'], db)
        if gold_fp is None:
            raise ValueError(f"Gold query failed for {r['db_id']}")
        matched = best_fp == gold_fp
    else:
        best_sql = ''
        matched = False

    sc_correct += int(matched)
    sc_results.append({
        'question': r['question'], 'db_id': r['db_id'],
        'difficulty': r['difficulty'],
        'pred_sql': best_sql, 'gold_sql': r['gold_sql'],
        'correct': matched, 'n_valid_candidates': len(candidates),
    })

sc_ex = sc_correct / len(bird_dev)
print(f'E1 SC@{N_SAMPLES} EX: {sc_ex:.4f} ({sc_correct}/{len(bird_dev)})')

with open(RESULTS_DIR / 'E1_sc16.json', 'w') as f:
    json.dump({'accuracy': sc_ex, 'total': len(bird_dev), 'correct': sc_correct,
               'model': MODEL_NAME, 'n_samples': N_SAMPLES, 'elapsed_s': sc_elapsed,
               'records': sc_results, 'split': 'heldout_legacy_dev', 'seed': 42, 'evaluator': 'bird-set-v1'}, f, indent=2)

## 4. Pilot — GPU throughput measurement

In [ ]:
import torch

if torch.cuda.is_available():
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    print(f'Peak VRAM: {peak_mb:.0f} MB')

print(f'\nThroughput summary:')
print(f'  Greedy: {len(prompts)/elapsed:.1f} queries/s')
print(f'  SC@{N_SAMPLES}: {len(prompts)/sc_elapsed:.1f} queries/s (x{N_SAMPLES} samples each)')
print(f'  SC@{N_SAMPLES} total samples: {len(prompts)*N_SAMPLES/sc_elapsed:.1f} samples/s')
greedy_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
sc_tokens = sum(len(c.token_ids) for o in sc_outputs for c in o.outputs)
print({'greedy_output_tokens_per_second': greedy_tokens / elapsed,
       'sc_output_tokens_per_second': sc_tokens / sc_elapsed})


## 5. Check author checkpoint (optional)

In [ ]:
# GitHub code repo and Hugging Face checkpoint IDs are different.
from huggingface_hub import model_info
for model_id in ('cycloneboy/SLM-SQL-0.5B', 'cycloneboy/SLM-SQL-1.5B',
                 'cycloneboy/CscSQL-Merge-Qwen2.5-Coder-1.5B-Instruct'):
    info = model_info(model_id)
    print({'checkpoint': model_id, 'revision': info.sha})
print('Use 09–11 to evaluate the author generator and merger together.')
